# ☁️ Облачный Запуск: UAV-CV-Navigator
Данный ноутбук предназначен для выполнения требования №24 ТЗ (*Запуск демо проекта по ссылке без установки дополнительных программ*).
Проект работает нативной среде Ubuntu 22.04 LTS (на серверах Google) с поддержкой Python 3.10+ и GPU.

In [ ]:
# ШАГ 1: Клонирование репозитория
# ВНИМАНИЕ: Замените ссылку ниже на URL вашего GitHub репозитория перед отправкой проверяющему!
!git clone https://github.com/ВАШ_АККАУНТ/ВАШ_РЕПОЗИТОРИЙ.git uav_cv_project
%cd uav_cv_project

In [ ]:
# ШАГ 2: Установка зависимостей (Требование ТЗ)
!pip install -r requirements.txt
!pip install ultralytics lapx numpy pandas matplotlib opencv-python

# Загрузка демо-видео (если оно не закоммичено в GitHub из-за размера)
!mkdir -p data
!wget -q https://github.com/intel-iot-devkit/sample-videos/raw/master/people-detection.mp4 -O data/test_video.mp4

In [ ]:
# ШАГ 3: Финальный запуск (В одну команду по ТЗ)
# Здесь применяется режим Zero Hardcode и используются конфиги YAML.
!export PYTHONPATH=$PYTHONPATH:$(pwd) && python src/pipeline.py --video data/test_video.mp4 --config configs/default.yaml --mode full

In [ ]:
# ШАГ 4: Визуализация сгенерированной телеметрии (Colab-only)
import pandas as pd
import matplotlib.pyplot as plt
import json

data = []
with open('data/output/telemetry.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))
        
df = pd.DataFrame(data)
df['obj_count'] = df['tracks'].apply(len)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))
ax1.plot(df['frame_id'], df['fps'], color='green')
ax1.set_title('Производительность (FPS)')
ax1.set_xlabel('Кадр')
ax1.set_ylabel('Кадров/Сек')
ax1.grid()

ax2.plot(df['frame_id'], df['obj_count'], color='purple')
ax2.set_title('Объекты в кадре (Навигация)')
ax2.set_xlabel('Кадр')
ax2.set_ylabel('Количество треков')
ax2.grid()

plt.show()

In [ ]:
# ШАГ 5: Покажем итоговое видео прямо в блокноте
from IPython.display import HTML
from base64 import b64encode

try:
    mp4 = open('data/output/result.mp4','rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f"""
    <video width=800 controls>
          <source src="{data_url}" type="video/mp4">
    </video>
    """))
except FileNotFoundError:
    print("Видео еще не сгенерировано. Проверьте логи ШАГА 3.")